In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "/kaggle/input/kunuz-cleaned/kunuz_final.csv"

# Load the latest version
df = pd.read_csv(file_path)

df = df[df['gender'] != 'unknown']
# not 17620 
len(df)

In [ ]:
import os, random, math, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

# -----------------
# Config
# -----------------
SEED = 42
BATCH_TRAIN = 32
BATCH_VAL = 32
EPOCHS = 20
LR = 2e-5
MAX_LEN = 512
MODEL_NAME = "elmurod1202/bertbek-news-big-cased"

device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
print('hi')

In [ ]:
from transformers import DataCollatorWithPadding
from sklearn.preprocessing import LabelEncoder

le_gender = LabelEncoder()
le_age = LabelEncoder()
le_industry = LabelEncoder()

df["gender_lbl"]   = le_gender.fit_transform(df["gender"])
df["year_lbl"] = le_age.fit_transform(df["year"])
df["topic_lbl"]  = le_industry.fit_transform(df["category"])


# -------- Dataset (no fixed padding here) --------
class MTDataset(Dataset):
    def __init__(self, df, max_len=MAX_LEN):
        self.texts = df["body"].astype(str).tolist()
        self.g = df["gender_lbl"].astype(int).tolist()
        self.y = df["year_lbl"].astype(int).tolist()
        self.t = df["topic_lbl"].astype(int).tolist()
        self.max_len = MAX_LEN
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = tok(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,   # cap length
            return_tensors="pt"        # <-- no padding="max_length"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["g"] = torch.tensor(self.g[idx], dtype=torch.float32)
        item["y"] = torch.tensor(self.y[idx], dtype=torch.long)
        item["t"] = torch.tensor(self.t[idx], dtype=torch.long)
        return item

# -------- Dynamic pad collator --------
collate_fn = DataCollatorWithPadding(tokenizer=tok, return_tensors="pt")

# -------- Split (keep your label encoders as you wrote) --------
train_df, val_df = train_test_split(
    df, test_size=0.15, random_state=SEED, stratify=df["gender_lbl"]
)

ds_tr, ds_va = MTDataset(train_df, max_len=MAX_LEN), MTDataset(val_df, max_len=MAX_LEN)

# Use workers + pin_memory for speed
dl_tr = DataLoader(
    ds_tr, batch_size=BATCH_TRAIN, shuffle=True,
    collate_fn=collate_fn, num_workers=4, pin_memory=True, persistent_workers=True
)
dl_va = DataLoader(
    ds_va, batch_size=BATCH_VAL, shuffle=False,
    collate_fn=collate_fn, num_workers=4, pin_memory=True, persistent_workers=True
)

num_year = df["year_lbl"].nunique()
num_topic = df["topic_lbl"].nunique()

In [ ]:
# -----------------
# Model
# -----------------
class MTLModel(nn.Module):
    def __init__(self, base, n_year, n_topic):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base)
        hid = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.head_g = nn.Linear(hid, 1)
        self.head_y = nn.Linear(hid, n_year)
        self.head_t = nn.Linear(hid, n_topic)
        self.bce = nn.BCEWithLogitsLoss()
        self.ce_y = nn.CrossEntropyLoss(label_smoothing=0.05)
        self.ce_t = nn.CrossEntropyLoss(label_smoothing=0.05)

    def forward(self, input_ids, attention_mask, g=None, y=None, t=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = self.dropout(out.last_hidden_state[:,0])  
        lg = self.head_g(h).squeeze(1)
        ly = self.head_y(h)
        lt = self.head_t(h)
        losses = None
        if g is not None:
            Lg = self.bce(lg, g)
            Ly = self.ce_y(ly, y)
            Lt = self.ce_t(lt, t)
            loss = Lg + Ly + Lt  
            losses = (loss, Lg.detach(), Ly.detach(), Lt.detach())
        return (lg, ly, lt), losses

# -----------------
# Training setup
# -----------------

model = MTLModel(MODEL_NAME, num_year, num_topic).to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
num_train_steps = EPOCHS * len(dl_tr)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=num_train_steps)

# -----------------
# Evaluation
# -----------------
def evaluate():
    model.eval()
    g_probs, g_true, y_pred, y_true, t_pred, t_true = [], [], [], [], [], []
    with torch.no_grad():
        for batch in dl_va:
            for k in ("input_ids","attention_mask"): batch[k] = batch[k].to(device)
            g, y, t = batch["g"].to(device), batch["y"].to(device), batch["t"].to(device)
            (lg, ly, lt), _ = model(batch["input_ids"], batch["attention_mask"])
            g_probs.extend(torch.sigmoid(lg).cpu().numpy())
            g_true.extend(g.cpu().numpy())
            y_pred.extend(ly.argmax(1).cpu().numpy()); y_true.extend(y.cpu().numpy())
            t_pred.extend(lt.argmax(1).cpu().numpy()); t_true.extend(t.cpu().numpy())
    g_probs = np.array(g_probs); g_true = np.array(g_true)
    g_pred = (g_probs >= 0.5).astype(int)
    return {
        "G_acc": accuracy_score(g_true, g_pred),
        "G_f1": f1_score(g_true, g_pred, average="macro"),
        "Y_acc": accuracy_score(y_true, y_pred),
        "Y_f1": f1_score(y_true, y_pred, average="macro"),
        "T_acc": accuracy_score(t_true, t_pred),
        "T_f1": f1_score(t_true, t_pred, average="macro"),
    }

In [ ]:
# import os
# import numpy as np
# from tqdm.auto import tqdm

# # where to save on Kaggle
# CKPT_DIR = "/kaggle/working/bertbek_mtl"
# os.makedirs(CKPT_DIR, exist_ok=True)

# for ep in range(1, EPOCHS + 1):
#     model.train()
#     tr_losses = []
#     pbar = tqdm(dl_tr, desc=f"Epoch {ep}", leave=False)

#     for batch in pbar:
#         for k in ("input_ids", "attention_mask"):
#             batch[k] = batch[k].to(device)

#         g = batch["g"].to(device)
#         y = batch["y"].to(device)
#         t = batch["t"].to(device)

#         # forward
#         (_, _, _), losses = model(
#             batch["input_ids"],
#             batch["attention_mask"],
#             g, y, t
#         )

#         loss = losses[0]
#         if loss.dim() > 0:      # e.g., DataParallel -> [num_devices]
#             loss = loss.mean()

#         optimizer.zero_grad()
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#         optimizer.step()
#         scheduler.step()

#         tr_losses.append(loss.item())
#         pbar.set_postfix({"loss": f"{loss.item():.4f}"})

#     # evaluation
#     metrics = evaluate()
#     comp = (metrics["G_f1"] + metrics["Y_f1"] + metrics["T_f1"]) / 3.0

#     print(
#         f"Epoch {ep:02d} | train {np.mean(tr_losses):.4f} | "
#         f"G {metrics['G_acc']:.3f}/{metrics['G_f1']:.3f} | "
#         f"Y {metrics['Y_acc']:.3f}/{metrics['Y_f1']:.3f} | "
#         f"T {metrics['T_acc']:.3f}/{metrics['T_f1']:.3f} | comp {comp:.3f}"
#     )

#     # save checkpoint for this epoch
#     ckpt_path = os.path.join(CKPT_DIR, f"bertbek_mtl_epoch{ep}.pt")
#     torch.save(model.state_dict(), ckpt_path)
#     print(f"Saved checkpoint: {ckpt_path}")

In [ ]:
import os
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score
import torch

# where to save on Kaggle
CKPT_DIR = "/kaggle/working/bertbek_mtl"
os.makedirs(CKPT_DIR, exist_ok=True)

for ep in range(1, EPOCHS + 1):
    model.train()
    tr_losses = []

    # --- NEW: collect TRAIN predictions/labels for metrics ---
    tr_g_true, tr_g_pred = [], []
    tr_y_true, tr_y_pred = [], []
    tr_t_true, tr_t_pred = [], []

    pbar = tqdm(dl_tr, desc=f"Epoch {ep}", leave=False)

    for batch in pbar:
        for k in ("input_ids", "attention_mask"):
            batch[k] = batch[k].to(device)

        g = batch["g"].to(device)
        y = batch["y"].to(device)
        t = batch["t"].to(device)

        # forward (logits + losses)
        (lg, ly, lt), losses = model(
            batch["input_ids"],
            batch["attention_mask"],
            g, y, t
        )

        loss = losses[0]
        if loss.dim() > 0:      # e.g., DataParallel -> [num_devices]
            loss = loss.mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        tr_losses.append(loss.item())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        # --- NEW: compute TRAIN preds for metrics (no extra forward) ---
        # If your gender head is CE with 2 logits: (B,2) -> argmax
        g_hat = (torch.sigmoid(lg).view(-1) >= 0.5).long()

        # If your gender head is BCE with 1 logit, use instead:
        # g_hat = (torch.sigmoid(lg).view(-1) >= 0.5).long()

        y_hat = ly.argmax(1)
        t_hat = lt.argmax(1)

        tr_g_true.append(g.detach().cpu()); tr_g_pred.append(g_hat.detach().cpu())
        tr_y_true.append(y.detach().cpu()); tr_y_pred.append(y_hat.detach().cpu())
        tr_t_true.append(t.detach().cpu()); tr_t_pred.append(t_hat.detach().cpu())

    # --- NEW: compute TRAIN metrics at end of epoch ---
    tr_g_true = torch.cat(tr_g_true).numpy()
    tr_g_pred = torch.cat(tr_g_pred).numpy()
    tr_y_true = torch.cat(tr_y_true).numpy()
    tr_y_pred = torch.cat(tr_y_pred).numpy()
    tr_t_true = torch.cat(tr_t_true).numpy()
    tr_t_pred = torch.cat(tr_t_pred).numpy()

    tr_metrics = {
        "G_acc": accuracy_score(tr_g_true, tr_g_pred),
        "G_f1":  f1_score(tr_g_true, tr_g_pred, average="macro"),
        "Y_acc": accuracy_score(tr_y_true, tr_y_pred),
        "Y_f1":  f1_score(tr_y_true, tr_y_pred, average="macro"),
        "T_acc": accuracy_score(tr_t_true, tr_t_pred),
        "T_f1":  f1_score(tr_t_true, tr_t_pred, average="macro"),
    }

    # evaluation (VAL)
    metrics = evaluate()
    comp = (metrics["G_f1"] + metrics["Y_f1"] + metrics["T_f1"]) / 3.0

    print(
        f"Epoch {ep:02d} | loss {np.mean(tr_losses):.4f} | "
        f"TR G {tr_metrics['G_acc']:.3f}/{tr_metrics['G_f1']:.3f} | "
        f"TR Y {tr_metrics['Y_acc']:.3f}/{tr_metrics['Y_f1']:.3f} | "
        f"TR T {tr_metrics['T_acc']:.3f}/{tr_metrics['T_f1']:.3f} | "
        f"VA G {metrics['G_acc']:.3f}/{metrics['G_f1']:.3f} | "
        f"VA Y {metrics['Y_acc']:.3f}/{metrics['Y_f1']:.3f} | "
        f"VA T {metrics['T_acc']:.3f}/{metrics['T_f1']:.3f} | "
        f"comp {comp:.3f}"
    )

    # save checkpoint for this epoch
    ckpt_path = os.path.join(CKPT_DIR, f"bertbek_mtl_epoch{ep}.pt")
    torch.save(model.state_dict(), ckpt_path)
    print(f"Saved checkpoint: {ckpt_path}")